# verify04: B型遷移モデルの発展① ― 履歴を入れる（＋長文モデル差し替え可）

前回挙げた先端アプローチのうち **「① 履歴(history)を入れる」** を実装。
B（遷移BERT）は今まで各ノード独立（マルコフ的）だったが、頭痛は 痛み→しびれ→振る舞い の直列なので
**前ノードの質問→答え を文脈に足す**と効く可能性がある。これを検証する。

| モデル | 入力 | 学習データ |
|---|---|---|
| **B0（履歴なし）** | 質問 + 採用ペア | 3ノード共有486例 |
| **B1（履歴あり）** | **[これまでの確認] 前ノードのQ→A** + 質問 + 採用ペア | 3ノード共有486例 |

- **学習時の履歴＝正解(gold)の前ノード答え**（teacher forcing）。
- **ノード別評価＝gold履歴**でteacher-forced比較。
- **トリアージ評価＝モデル自身の予測で履歴を作りながら**決定木を辿る（実運用に近い逐次判定）。
- **長文モデルの差し替え**：`BASE_MODEL` を `sbintuitions/modernbert-ja-130m` 等にすれば、
  履歴を長く入れても切れにくい（長文レバー）。

> トリアージ/遷移・履歴の関数以外（学習ループ等）は `HeadacheBERT_painful_Finetuning.ipynb` 流用。

# 1. セットアップ（GPUは git clone / ローカルはそのまま）

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'
IN_COLAB = 'google.colab' in sys.modules


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', 'input_pairs.csv')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    # ModernBERT-Ja を使うなら新しめの transformers が要るので最新を入れる
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'transformers', 'sentencepiece', 'fugashi', 'unidic-lite',
                    'accelerate', 'pyyaml'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', 'input_pairs.csv')
YAML_PATH = os.path.join(REPO_DIR, 'transition_diagram', 'protocol.yaml')
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

# 2. インポート & 設定

In [ ]:
import time, random
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import fugashi
import transformers
transformers.logging.set_verbosity_error()   # 余計なロードレポート/警告を抑制

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

# ===== 設定（B0/B1 共通）=====
# 長文レバー：下を差し替えると長文モデルになる（履歴を長く入れても切れにくい）
#   'cl-tohoku/bert-base-japanese-v3'   ← 既定（512トークン）
#   'sbintuitions/modernbert-ja-130m'   ← ModernBERT-Ja（長文・高速）。MAX_LENGTHを伸ばすと効く
BASE_MODEL = 'cl-tohoku/bert-base-japanese-v3'
MAX_LENGTH = 512
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 8
N_FOLDS = 5
USE_STOPWORDS = False
NUM_LABELS = 3
SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f'BASE_MODEL={BASE_MODEL} MAX_LENGTH={MAX_LENGTH} epochs={NUM_EPOCHS} folds={N_FOLDS}')

# 3. データ・決定木・採用ペア（流用＋遷移用）

In [ ]:
df = pd.read_csv(CSV_PATH)
print('rows:', len(df), '/ patients:', df['id'].nunique())

import yaml
_proto = yaml.safe_load(open(YAML_PATH, encoding='utf-8'))
_h = next(p for p in _proto['protocols'] if p['id'] == 'headache')
fallback_triage = _h['fallback']['if_all_symptom_questions_negative']


def _is_branch(n):
    return 'choices' in n and not n.get('metadata_only', False)


def _parse_choice(c):
    if c.get('triage'):
        return {'action': 'terminal', 'triage': c['triage']}
    if c.get('next'):
        return {'action': 'next', 'next_id': c['next']}
    return {'action': 'fallback', 'triage': fallback_triage}


branch_table = [{'id': n['id'], 'question': n['question'],
                 'choices': [_parse_choice(c) for c in n['choices']]}
                for n in _h['nodes'] if _is_branch(n)]
branch_ids = {b['id'] for b in branch_table}
triage_decode = {0: 'R3', 1: 'R2', 2: 'Y2'}
LABEL_TEXT = {0: 'はい', 1: 'いいえ', 2: '不明'}

NODES = [
    {'key': '痛み',   'adopt': '採用ペア_ひし形_全通り_頭痛_1', 'label': '痛み',   'bid': 'headache_sudden_severe'},
    {'key': 'しびれ', 'adopt': '採用ペア_ひし形_全通り_頭痛_2', 'label': 'しびれ', 'bid': 'headache_numbness_paralysis'},
    {'key': '振る舞い', 'adopt': '採用ペア_ひし形_全通り_頭痛_3', 'label': '振る舞い', 'bid': 'headache_abnormal_behavior'},
]
_qmap = {b['id']: b['question'] for b in branch_table}
for n in NODES:
    n['question'] = _qmap[n['bid']]
NODE_BY_BID = {n['bid']: n for n in NODES}
print('branch_table:', [b['id'] for b in branch_table], '| fallback:', fallback_triage)

# ストップワード除去（painful 流用）
_tagger = fugashi.Tagger()
STOPWORD_EXTRA_WORDS = set(['の', 'は', 'を', 'に', 'が', 'で', 'と', 'も', 'から', 'より',
                            'へ', 'や', 'など', 'ので', 'けど', 'けれど', '、', '。', 'です', 'ます'])


def remove_stopwords(text: str) -> str:
    return ''.join(w.surface for w in _tagger(text) if w.surface not in STOPWORD_EXTRA_WORDS)


# 患者×ノードの採用ペア（質問は別で付ける）
_adopt_cache = {}
def adopted_text(pid, node):
    key = (pid, node['key'])
    if key not in _adopt_cache:
        g = df[df['id'] == pid]
        ad = g[g[node['adopt']] == True]
        t = ' '.join(ad['ペア'].astype(str).tolist()) if len(ad) else '(発話なし)'
        _adopt_cache[key] = remove_stopwords(t) if USE_STOPWORDS else t
    return _adopt_cache[key]

# 4. painful 流用：学習・モデル部品

In [ ]:
_tok_cache: Dict[str, 'AutoTokenizer'] = {}
def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


class PainTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts, self.labels = list(texts), list(labels)
        self.tokenizer, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length,
                             padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def build_model(name):
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=NUM_LABELS,
                                                               trust_remote_code=True)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        print(f'  [警告] 語彙数{len(tok)}!=vocab{model.config.vocab_size} → resize')
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


def train_b_model(texts, labels):
    """painful の学習ループそのままで、与えたテキスト・ラベルでBERTを1本学習。"""
    set_seed(SEED)
    tok = get_tokenizer(BASE_MODEL)
    model = build_model(BASE_MODEL)
    loader = DataLoader(PainTextDataset(texts, labels, tok, MAX_LENGTH),
                        batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    model.train()
    for epoch in range(NUM_EPOCHS):
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
    model.eval()   # ★推論前にevalへ（dropoutを切る）。これが無いと予測がノイズで悪化する
    return model, tok


@torch.no_grad()
def predict_one(model, tok, text):
    model.eval()   # 念のため（eval固定）
    enc = tok([text], truncation=True, max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    return int(model(**enc).logits.argmax(dim=-1).cpu())


print('学習部品を定義（painful流用＋長文対応）')

# 5. ★履歴・遷移の中核★（ここが新規）

- `render_history`：[これまでの確認] 前ノードの「質問→答え」を文字列化。
- `build_context`：`use_history` に応じて 履歴 + 質問 + 採用ペア を作る。
- `build_train_examples`：**gold履歴**で全(患者×3ノード)の学習例を作る。
- `predict_nodes_teacherforced`：**gold履歴**でノード別予測（teacher-forced 比較用）。
- `predict_triage_traverse`：**自分の予測で履歴を作りながら**決定木を辿り最終トリアージ。

In [ ]:
def render_history(history):
    if not history:
        return ''
    body = ' ; '.join(f'{q}→{a}' for q, a in history)
    return f'[これまでの確認] {body} '


def build_context(pid, node, history, use_history):
    head = render_history(history) if use_history else ''
    return head + node['question'] + ' ' + adopted_text(pid, node)


def build_train_examples(train_ids, use_history):
    """gold履歴で全(患者×3ノード)を例化（full coverage）。"""
    texts, labels = [], []
    for pid in train_ids:
        history = []
        for node in NODES:
            texts.append(build_context(pid, node, history, use_history))
            lab = true_node[node['key']][pid]
            labels.append(lab)
            history.append((node['question'], LABEL_TEXT[lab]))  # 正解の前ノード答えを履歴に
    return texts, labels


def predict_nodes_teacherforced(model, tok, test_ids, use_history):
    """gold履歴を使ってノードごとに予測（B0/B1 を同条件で比較するため）。"""
    pred = {n['key']: {} for n in NODES}
    for pid in test_ids:
        history = []
        for node in NODES:
            ctx = build_context(pid, node, history, use_history)
            pred[node['key']][pid] = predict_one(model, tok, ctx)
            history.append((node['question'], LABEL_TEXT[true_node[node['key']][pid]]))
    return pred


def predict_triage_traverse(model, tok, pid, use_history):
    """自分の予測で履歴を作りながら決定木を辿り、最終トリアージを返す。"""
    history = []
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        node = NODE_BY_BID[b['id']]
        ctx = build_context(pid, node, history, use_history)
        p = predict_one(model, tok, ctx)
        ch = b['choices'][p]
        history.append((b['question'], LABEL_TEXT[p]))  # 予測を履歴に
        if ch['action'] in ('terminal', 'fallback'):
            return ch['triage']
        if ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        return fallback_triage


print('履歴・遷移の中核を定義。')

# 6. fold分割・正解・サニティ

In [ ]:
patients = np.array(sorted(df['id'].unique()))
triage_strat = np.array([int(df[df['id'] == p]['トリアージ'].iloc[0]) for p in patients])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = [(patients[tr], patients[te]) for tr, te in skf.split(patients, triage_strat)]

true_node = {n['key']: {p: int(df[df['id'] == p][n['label']].iloc[0]) for p in patients} for n in NODES}
true_triage = {p: triage_decode[int(df[df['id'] == p]['トリアージ'].iloc[0])] for p in patients}
print(f'{N_FOLDS}-fold。test患者数:', [len(te) for _, te in FOLDS])

# サニティ：gold答えで辿ると真トリアージに一致するはず
def _gold_traverse(pid):
    history = []
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        lab = true_node[NODE_BY_BID[b['id']]['key']][pid]
        ch = b['choices'][lab]
        if ch['action'] in ('terminal', 'fallback'):
            return ch['triage']
        if ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        return fallback_triage


_acc = np.mean([_gold_traverse(p) == true_triage[p] for p in patients])
print(f'[sanity] gold答えで辿ったトリアージ一致率 = {_acc:.3f}（1.000ならOK）')

# 7. 学習＆評価：B0（履歴なし） vs B1（履歴あり）

In [ ]:
def run_B(use_history, tag):
    predN = {n['key']: {} for n in NODES}
    predTri = {}
    for i, (tr_ids, te_ids) in enumerate(FOLDS):
        texts, labels = build_train_examples(tr_ids, use_history)
        model, tok = train_b_model(texts, labels)
        # ノード別（gold履歴・teacher-forced）
        pn = predict_nodes_teacherforced(model, tok, te_ids, use_history)
        for n in NODES:
            predN[n['key']].update(pn[n['key']])
        # トリアージ（予測履歴で逐次トラバース）
        for pid in te_ids:
            predTri[pid] = predict_triage_traverse(model, tok, pid, use_history)
        print(f'  [{tag}] fold{i} done')
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return predN, predTri


print('=== B0（履歴なし）===')
predN0, predTri0 = run_B(False, 'B0')
print('=== B1（履歴あり）===')
predN1, predTri1 = run_B(True, 'B1')
print('完了')

# 8. 結果①：ノード別 accuracy / macro-F1（B0 vs B1）

In [ ]:
def node_scores(pred_for_node, key):
    yt = [true_node[key][p] for p in patients]
    yp = [pred_for_node[p] for p in patients]
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


rows = []
for n in NODES:
    k = n['key']
    a0, f0 = node_scores(predN0[k], k)
    a1, f1v = node_scores(predN1[k], k)
    rows.append({'ノード': k, 'B0 acc': f'{a0:.3f}', 'B1 acc': f'{a1:.3f}',
                 'Δacc': f'{a1 - a0:+.3f}', 'B0 F1': f'{f0:.3f}', 'B1 F1': f'{f1v:.3f}'})
node_table = pd.DataFrame(rows)
print('===== ノード別（B0=履歴なし / B1=履歴あり）=====')
display(node_table)
node_table.to_csv(os.path.join(OUT_DIR, 'verify04_node.csv'), index=False, encoding='utf-8-sig')

# 9. 結果②：最終トリアージ（予測履歴で決定木を辿る）B0 vs B1

In [ ]:
def triage_scores(predTri):
    yt = [true_triage[p] for p in patients]
    yp = [predTri[p] for p in patients]
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


tri_rows = []
for name, pt in [('B0 履歴なし', predTri0), ('B1 履歴あり', predTri1)]:
    a, f = triage_scores(pt)
    tri_rows.append({'モデル': name, 'トリアージ acc': f'{a:.3f}', 'トリアージ macro-F1': f'{f:.3f}'})
tri_table = pd.DataFrame(tri_rows)
print('===== 最終トリアージ R3/R2/Y2 =====')
display(tri_table)
tri_table.to_csv(os.path.join(OUT_DIR, 'verify04_triage.csv'), index=False, encoding='utf-8-sig')
print('saved: verify04_node.csv, verify04_triage.csv')

# 10. まとめ（読み方）

- **B0 vs B1**：履歴（前ノードの質問→答え）を入れる効果。Δが+なら履歴が効く、≈0/−なら不要（max_len圧迫・過学習で悪化もある）。
- **ノード別はgold履歴**（teacher-forced）で公平比較、**トリアージは予測履歴**で逐次トラバース（実運用に近い）。
- **長文レバー**：`BASE_MODEL` を `sbintuitions/modernbert-ja-130m` に変え `MAX_LENGTH` を伸ばすと、履歴を長く入れても切れにくい。B1で特に効くか確認できる。
- 採用ペア・fold・ハイパラは B0/B1 共通。学習ループ等は painful 流用、履歴・遷移部分だけ新規。